# Notebook 50: Simulator

This notebook demonstrates testing and simulation utilities using `multigen.simulator`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `DryRunSimulator` — set mocks for nodes, capture call trace
- `MockInjector` — inject a fake response on a specific call number
- `LoadSimulator` — concurrent load test with P95 and throughput report
- `CostEstimator` — per-node token cost forecast with render
- `GraphDiffVisualiser` — ASCII diff between two graph definitions

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.simulator`

In [ ]:
import asyncio
import statistics
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

print('Imports ready.')


## Section 1 — DryRunSimulator

In [ ]:
@dataclass
class DryRunReport:
    node_calls: List[str]
    ctx_trace: Dict[str, Any]


class DryRunSimulator:
    """Runs a pipeline of (node_id, fn) pairs with optional mocked nodes."""

    def __init__(self):
        self._mocks: Dict[str, Any] = {}

    def set_mock(self, node_id: str, return_value: Any):
        self._mocks[node_id] = return_value

    def run(self, pipeline: List[Tuple[str, Callable]], initial_input: Any) -> DryRunReport:
        ctx = initial_input
        node_calls = []
        ctx_trace = {'__input__': initial_input}

        for node_id, fn in pipeline:
            if node_id in self._mocks:
                ctx = self._mocks[node_id]
                node_calls.append(f'{node_id}(MOCKED)')
            else:
                ctx = fn(ctx)
                node_calls.append(f'{node_id}(REAL)')
            ctx_trace[node_id] = ctx

        return DryRunReport(node_calls=node_calls, ctx_trace=ctx_trace)


# Define a simple mock pipeline
def fetch_data(query: str) -> Dict:
    return {'raw': f'data for query: {query}', 'count': 5}

def process_data(data: Dict) -> Dict:
    return {'processed': data['raw'].upper(), 'count': data['count']}

def format_output(data: Dict) -> str:
    return f'Result: {data["processed"]} ({data["count"]} items)'


pipeline = [
    ('fetch',   fetch_data),
    ('process', process_data),
    ('format',  format_output),
]

sim = DryRunSimulator()
sim.set_mock('fetch', {'raw': 'MOCK FETCHED DATA', 'count': 99})

report = sim.run(pipeline, initial_input='quarterly revenue')

print('DryRunReport:')
print(f'  node_calls : {report.node_calls}')
print(f'  ctx_trace  :')
for k, v in report.ctx_trace.items():
    print(f'    {k}: {v}')


## Section 2 — MockInjector

In [ ]:
class MockInjector:
    """
    Wraps a real agent function and injects a fake return value
    on a specific call number (1-indexed).
    """

    def __init__(self, real_fn: Callable, inject_on: int, inject_value: Any):
        self.real_fn = real_fn
        self.inject_on = inject_on
        self.inject_value = inject_value
        self._call_count = 0

    def __call__(self, *args, **kwargs) -> Any:
        self._call_count += 1
        if self._call_count == self.inject_on:
            print(f'  call #{self._call_count}: INJECTED -> {self.inject_value!r}')
            return self.inject_value
        result = self.real_fn(*args, **kwargs)
        print(f'  call #{self._call_count}: REAL     -> {result!r}')
        return result


def real_classifier(text: str) -> str:
    return 'positive' if 'good' in text.lower() else 'negative'


injected_agent = MockInjector(
    real_fn=real_classifier,
    inject_on=2,
    inject_value='ERROR: model_timeout',
)

inputs = ['This is good news', 'Something went wrong', 'Great results today']

print('MockInjector demo (inject on call #2):')
for text in inputs:
    injected_agent(text)


## Section 3 — LoadSimulator

In [ ]:
@dataclass
class LoadScenario:
    concurrency: int
    total_requests: int
    timeout_ms: float = 5000


@dataclass
class LoadReport:
    total_requests: int
    success_count: int
    error_count: int
    latencies_ms: List[float]

    @property
    def p50_ms(self) -> float:
        s = sorted(self.latencies_ms)
        return round(s[int(0.50 * len(s))], 3) if s else 0.0

    @property
    def p95_ms(self) -> float:
        s = sorted(self.latencies_ms)
        return round(s[int(0.95 * len(s))], 3) if s else 0.0

    @property
    def throughput_rps(self) -> float:
        if not self.latencies_ms:
            return 0.0
        total_s = sum(self.latencies_ms) / 1000
        return round(self.total_requests / total_s, 2) if total_s > 0 else 0.0


class LoadSimulator:
    def __init__(self, scenario: LoadScenario):
        self.scenario = scenario

    async def run(self, async_fn: Callable) -> LoadReport:
        latencies = []
        errors = 0

        sem = asyncio.Semaphore(self.scenario.concurrency)

        async def _single_request(i: int):
            nonlocal errors
            async with sem:
                t0 = time.perf_counter()
                try:
                    await async_fn(i)
                except Exception:
                    errors += 1
                    return
                elapsed_ms = (time.perf_counter() - t0) * 1000
                latencies.append(elapsed_ms)

        tasks = [_single_request(i) for i in range(self.scenario.total_requests)]
        await asyncio.gather(*tasks)

        return LoadReport(
            total_requests=self.scenario.total_requests,
            success_count=len(latencies),
            error_count=errors,
            latencies_ms=latencies,
        )


# Fast async mock function (simulates 1-5ms work)
import random
rng = random.Random(42)

async def fast_mock_fn(request_id: int):
    await asyncio.sleep(rng.uniform(0.001, 0.005))  # 1-5ms
    return f'result_{request_id}'


scenario = LoadScenario(concurrency=10, total_requests=50)
load_sim = LoadSimulator(scenario)

report = await load_sim.run(fast_mock_fn)

print(f'LoadReport:')
print(f'  total_requests  : {report.total_requests}')
print(f'  success_count   : {report.success_count}')
print(f'  error_count     : {report.error_count}')
print(f'  p50_ms          : {report.p50_ms}')
print(f'  p95_ms          : {report.p95_ms}')
print(f'  throughput_rps  : {report.throughput_rps}')


## Section 4 — CostEstimator

In [ ]:
# Cost per 1000 tokens (USD)
MODEL_COSTS = {
    'gpt-4':       {'input': 0.03,   'output': 0.06},
    'gpt-3.5':     {'input': 0.0015, 'output': 0.002},
    'claude-3':    {'input': 0.015,  'output': 0.075},
    'local-llama': {'input': 0.0,    'output': 0.0},
}


@dataclass
class NodeCostSpec:
    node_id: str
    model: str
    input_tokens: int
    output_tokens: int
    calls_per_run: int = 1


@dataclass
class CostForecast:
    node_id: str
    model: str
    cost_per_call_usd: float
    cost_per_run_usd: float


class CostEstimator:
    def __init__(self):
        self._nodes: List[NodeCostSpec] = []

    def add_node(self, node_id: str, model: str, input_tokens: int,
                 output_tokens: int, calls_per_run: int = 1):
        self._nodes.append(NodeCostSpec(
            node_id=node_id, model=model,
            input_tokens=input_tokens, output_tokens=output_tokens,
            calls_per_run=calls_per_run,
        ))

    def _node_cost(self, spec: NodeCostSpec) -> float:
        rates = MODEL_COSTS.get(spec.model, {'input': 0.0, 'output': 0.0})
        return round(
            (spec.input_tokens  / 1000) * rates['input'] +
            (spec.output_tokens / 1000) * rates['output'],
            6
        )

    def forecast(self) -> List[CostForecast]:
        result = []
        for spec in self._nodes:
            cpp = self._node_cost(spec)
            result.append(CostForecast(
                node_id=spec.node_id,
                model=spec.model,
                cost_per_call_usd=cpp,
                cost_per_run_usd=round(cpp * spec.calls_per_run, 6),
            ))
        return result

    def render(self) -> str:
        forecasts = self.forecast()
        total = sum(f.cost_per_run_usd for f in forecasts)
        lines = [
            f'{"NODE":<20} {"MODEL":<15} {"$/call":>10} {"$/run":>10}',
            '-' * 60,
        ]
        for f in forecasts:
            lines.append(
                f'{f.node_id:<20} {f.model:<15} {f.cost_per_call_usd:>10.6f} {f.cost_per_run_usd:>10.6f}'
            )
        lines.append('-' * 60)
        lines.append(f'{"TOTAL":<20} {"":<15} {"":>10} {total:>10.6f}')
        return '\n'.join(lines)


estimator = CostEstimator()
estimator.add_node('query_router',  'gpt-3.5',     input_tokens=200,   output_tokens=50,   calls_per_run=1)
estimator.add_node('retriever',     'local-llama', input_tokens=1000,  output_tokens=0,    calls_per_run=3)
estimator.add_node('reranker',      'gpt-3.5',     input_tokens=2000,  output_tokens=100,  calls_per_run=1)
estimator.add_node('answer_gen',    'gpt-4',       input_tokens=3000,  output_tokens=500,  calls_per_run=1)

print(estimator.render())


## Section 5 — GraphDiffVisualiser

In [ ]:
@dataclass
class GraphDefinition:
    name: str
    nodes: List[str]
    edges: List[Tuple[str, str]]  # (from, to)


class GraphDiffVisualiser:
    def __init__(self, graph_a: GraphDefinition, graph_b: GraphDefinition):
        self.graph_a = graph_a
        self.graph_b = graph_b

    def diff(self) -> Dict:
        nodes_a = set(self.graph_a.nodes)
        nodes_b = set(self.graph_b.nodes)
        edges_a = set(self.graph_a.edges)
        edges_b = set(self.graph_b.edges)

        return {
            'added_nodes':   sorted(nodes_b - nodes_a),
            'removed_nodes': sorted(nodes_a - nodes_b),
            'common_nodes':  sorted(nodes_a & nodes_b),
            'added_edges':   sorted(edges_b - edges_a),
            'removed_edges': sorted(edges_a - edges_b),
            'common_edges':  sorted(edges_a & edges_b),
        }

    def render(self) -> str:
        d = self.diff()
        lines = [
            f'=== Graph Diff: {self.graph_a.name!r} -> {self.graph_b.name!r} ===',
            '',
            'NODES:',
        ]
        for n in d['added_nodes']:
            lines.append(f'  + {n}')
        for n in d['removed_nodes']:
            lines.append(f'  - {n}')
        for n in d['common_nodes']:
            lines.append(f'    {n}')

        lines.append('')
        lines.append('EDGES:')
        for (src, dst) in d['added_edges']:
            lines.append(f'  + {src} --> {dst}')
        for (src, dst) in d['removed_edges']:
            lines.append(f'  - {src} --> {dst}')
        for (src, dst) in d['common_edges']:
            lines.append(f'    {src} --> {dst}')

        lines.append('')
        lines.append(
            f'Summary: +{len(d["added_nodes"])} nodes, '
            f'-{len(d["removed_nodes"])} nodes, '
            f'+{len(d["added_edges"])} edges, '
            f'-{len(d["removed_edges"])} edges'
        )
        return '\n'.join(lines)


# Graph v1
graph_v1 = GraphDefinition(
    name='pipeline_v1',
    nodes=['fetch', 'process', 'format', 'output'],
    edges=[('fetch', 'process'), ('process', 'format'), ('format', 'output')],
)

# Graph v2: add 'validate' node + edge, remove edge format->output, add reorder->output
graph_v2 = GraphDefinition(
    name='pipeline_v2',
    nodes=['fetch', 'process', 'validate', 'format', 'output'],
    edges=[
        ('fetch', 'process'),
        ('process', 'validate'),   # new edge
        ('validate', 'format'),    # new edge
        ('format', 'output'),      # kept
        # ('process', 'format') removed
    ],
)

visualiser = GraphDiffVisualiser(graph_v1, graph_v2)
print(visualiser.render())
